# Text-to-SQL inference through Google Drive and ngrok

Run this notebook in Google Colab. Put the project source files, `best_model.pt`, and `data/sql_sp.model` in one Google Drive folder.

In [ ]:
!pip -q install pyngrok flask requests sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Code is cloned from GitHub. Only model artifacts are read from Drive.
REPO_URL = 'https://github.com/MuhammadShaafImran/text-2-SQL-scratch.git'
REPO_DIR = '/content/text-2-SQL-scratch'
DRIVE_MODEL_DIR = '/content/drive/MyDrive/text-2-SQL-scratch'
CHECKPOINT_PATH = f'{DRIVE_MODEL_DIR}/best_model.pt'
TOKENIZER_PATH = f'{DRIVE_MODEL_DIR}/sql_sp.model'

import os
import sys
import subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
required_paths = [Path(CHECKPOINT_PATH), Path(TOKENIZER_PATH)]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing Drive model artifacts:\n' + '\n'.join(missing))
print(f'Using code from: {REPO_URL}')
print(f'Using model artifacts from: {DRIVE_MODEL_DIR}')

In [ ]:
import torch
from model.transformer import Transformer
from starter.embeddings import TokenEmbedding
from starter.tokenizer import load_tokenizer, PAD_ID
from decode import greedy_decode, parse_prediction, query_to_sql

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sp = load_tokenizer(TOKENIZER_PATH)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
config = checkpoint['config']
vocab_size = sp.get_piece_size()
model = Transformer(
    num_layers=config['num_layers'],
    d_model=config['d_model'],
    heads=config['heads'],
    token_embedding=TokenEmbedding(vocab_size, config['d_model'], PAD_ID),
    ffn_dim=config['d_ff'],
    max_len=512,
    dropout=config['dropout'],
    vocab_size=vocab_size,
).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Model loaded on {DEVICE}; vocabulary size: {vocab_size}')

In [ ]:
from flask import Flask, jsonify, request
import threading

api = Flask('text_to_sql_inference')

def build_prompt(question, columns):
    column_tokens = ' '.join(
        f'<c{index}> {column}' for index, column in enumerate(columns)
    )
    return f'{question.strip()} <sep> {column_tokens}'.lower()

@api.after_request
def add_cors_headers(response):
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = 'Content-Type'
    response.headers['Access-Control-Allow-Methods'] = 'POST, OPTIONS'
    return response

@api.route('/api/query', methods=['OPTIONS'])
def query_options():
    return ('', 204)

@api.post('/api/query')
def generate_query():
    payload = request.get_json(silent=True) or {}
    question = payload.get('question')
    columns = payload.get('columns')
    if not isinstance(question, str) or not question.strip():
        return jsonify({'error': 'question is required'}), 400
    if not isinstance(columns, str) or not columns.strip():
        return jsonify({'error': 'columns is required'}), 400
    header = [column.strip() for column in columns.split(',') if column.strip()]
    if not header:
        return jsonify({'error': 'columns must contain at least one column'}), 400
    try:
        source_ids = sp.encode(build_prompt(question, header), out_type=int)
        src_ids = torch.tensor([source_ids], dtype=torch.long, device=DEVICE)
        with torch.inference_mode():
            _, generated = greedy_decode(model, src_ids, sp, DEVICE)
        parsed = parse_prediction(generated)
        sql = query_to_sql(parsed, header) if parsed is not None else None
    except Exception as error:
        api.logger.exception('Local model inference failed')
        return jsonify({'error': f'Inference failed: {error}'}), 500
    return jsonify({'sql': sql or generated, 'query': generated,
                    'message': 'Generated by Drive-hosted model'})

server_thread = threading.Thread(
    target=lambda: api.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False),
    daemon=True,
)
server_thread.start()
print('Flask inference server started on port 5000')

In [ ]:
from pyngrok import ngrok
import getpass

NGROK_AUTHTOKEN = os.environ.get('NGROK_AUTHTOKEN')
if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = getpass.getpass('Paste your ngrok authtoken: ')
    os.environ['NGROK_AUTHTOKEN'] = NGROK_AUTHTOKEN
ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(5000, 'http')
PUBLIC_API_URL = tunnel.public_url
print(f'Public API URL: {PUBLIC_API_URL}')
print(f'Use this endpoint in the web app: {PUBLIC_API_URL}/api/query')

In [ ]:
import requests
test_payload = {
    'question': 'what position does the player from Butler CC play?',
    'columns': 'player, no., nationality, position, years in toronto, school/club team',
}
test_response = requests.post(f'{PUBLIC_API_URL}/api/query', json=test_payload, timeout=120)
print(test_response.status_code)
print(test_response.json())

Keep this Colab runtime running while the URL is in use. To stop the tunnel, run `ngrok.kill()`.